# **SOLUCIÓN DEL RETO 3 · Spark con DataFrames**
> Notebook realizado en **`Google Collab`**

> Se usa como fuente de datos el archivo proporcionado **`churn.csv`**.

> Apartados/**secuencia** seguida:

0. Instalación de `PySpark`
1. Subida de archivos a `Google Collab`
2. Localización automática del archivo .csv
3. **Punto 1 del reto** --> Creación de una sesión de `SparkSession`
4. **Punto 2 del reto** --> Lectura del fichero csv con `read()`
5. **Punto 3 del reto** --> Impresión del esquema del dataframe con `printSquema()`
6. Vista previa de los datos con `show()`
7. **Punto 4 del reto** --> Selección de columnas con `select()` y transformación con `col()`
8. **Punto 5 del reto** --> Filtrado de datos mediante `filter()`
9. **Punto 6 del reto** --> Primera agregación con `count()`
10. **Punto 7 del reto** --> Segunda agregación con `count()`
11. **Punto 8 del reto** --> Tercera agregación con `sum()`
12. **Punto 9 del reto** --> Cuarta agregación con `sum()`

**0. INSTALACIÓN DE PYSPARK**
> Este paso no es necesario, puesto que Collab ya lo carga, pero es una buena práctica

In [3]:
!pip -q install pyspark

**1. SUBIDA DE ARCHIVOS A GOOGLE COLLAB**

> El siguiente código abre un cuadro de diálogo para seleccionar el archivo `churn.csv` que contiene los datos de trabajo

In [4]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Archivos subidos correctamente.")
except Exception:
    print("Si no estás en Google Colab, coloca los CSV manualmente dentro de /content")

Saving churn.csv to churn.csv
Archivos subidos correctamente.


In [5]:
import os
import glob

print("Directorio actual:", os.getcwd())
print("CSV en /content:", glob.glob("/content/*.csv"))

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.lower().endswith(".csv"):
            print("Encontrado:", os.path.join(root, f))

Directorio actual: /content
CSV en /content: ['/content/churn.csv']
Encontrado: /content/churn.csv
Encontrado: /content/sample_data/california_housing_test.csv
Encontrado: /content/sample_data/mnist_test.csv
Encontrado: /content/sample_data/california_housing_train.csv
Encontrado: /content/sample_data/mnist_train_small.csv


**2. LOCALIZACIÓN AUTOMÁTICA DEL ARCHIVO CSV**
> Esta celda busca dentro de `/content` un archivo que contenga `churn` en el nombre.

In [6]:
import glob
import os

csv_files = glob.glob("/content/*.csv")

def localizar_csv(palabra_clave):
    candidatos = [
        f for f in csv_files
        if palabra_clave.lower() in os.path.basename(f).lower()
    ]
    if not candidatos:
        raise FileNotFoundError(
            f"No se ha encontrado ningún CSV que contenga '{palabra_clave}' en /content"
        )
    return candidatos[0]

churn_path = localizar_csv("churn")

print("Ruta churn.csv:", churn_path)

Ruta churn.csv: /content/churn.csv


**3. CREACIÓN DE LA SESIÓN DE SPARK (SPARKSESSION)** --> ***`Punto 1 del Reto`***
> Se crea la sesión de trabajo en PySpark.

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Reto3_BigDataIA_UAX_DAM1_churn")
    .getOrCreate()
)

spark

**4. LECTURA DEL FICHERO CSV** --> ***`Punto 2 del Reto`***
- Cargamos la información del archivo `churn.csv` con `read()`
- Se genera un dataframe a partir del mismo
- Mostramos el número de registros que contiene con `print()` y `count()`

In [8]:
churn_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(churn_path)
)

print("Número de registros en churn_df:", churn_df.count())

Número de registros en churn_df: 5000


**5. IMPRESION DEL ESQUEMA DEL DATAFRAME** --> ***`Punto 3 del Reto`***
> Imprimimos el esquema (columnas y tipo de dato) con `printSquema()`

In [9]:
print("Esquema de churn_df")
churn_df.printSchema()

print("\nEsquema de churn_df")
churn_df.printSchema()

Esquema de churn_df
root
 |-- State: string (nullable = true)
 |-- Account_Length: integer (nullable = true)
 |-- Area_Code: integer (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Intl_Plan: string (nullable = true)
 |-- Vmail_Plan: string (nullable = true)
 |-- Vmail_Message: integer (nullable = true)
 |-- Day_Mins: double (nullable = true)
 |-- Day_Calls: integer (nullable = true)
 |-- Day_Charge: double (nullable = true)
 |-- Eve_Mins: double (nullable = true)
 |-- Eve_Calls: integer (nullable = true)
 |-- Eve_Charge: double (nullable = true)
 |-- Night_Mins: double (nullable = true)
 |-- Night_Calls: integer (nullable = true)
 |-- Night_Charge: double (nullable = true)
 |-- Intl_Mins: double (nullable = true)
 |-- Intl_Calls: integer (nullable = true)
 |-- Intl_Charge: double (nullable = true)
 |-- CustServ_Calls: integer (nullable = true)
 |-- Churn: string (nullable = true)


Esquema de churn_df
root
 |-- State: string (nullable = true)
 |-- Account_Length: integer (

**6. VISTA PREVIA DE LOS DATOS**
> Mostramos los 5 primeros registros con `show()`

In [10]:
print("Primeras filas de churn_df")
churn_df.show(5, truncate=False)

Primeras filas de churn_df
+-----+--------------+---------+--------+---------+----------+-------------+------------------+---------+-----------------+------------------+---------+------------------+------------------+-----------+------------------+------------------+----------+------------------+--------------+------+
|State|Account_Length|Area_Code|Phone   |Intl_Plan|Vmail_Plan|Vmail_Message|Day_Mins          |Day_Calls|Day_Charge       |Eve_Mins          |Eve_Calls|Eve_Charge        |Night_Mins        |Night_Calls|Night_Charge      |Intl_Mins         |Intl_Calls|Intl_Charge       |CustServ_Calls|Churn |
+-----+--------------+---------+--------+---------+----------+-------------+------------------+---------+-----------------+------------------+---------+------------------+------------------+-----------+------------------+------------------+----------+------------------+--------------+------+
|PA   |163           |806      |403-2562|no       |yes       |300          |8.162204021739099 

**7. SELECCIÓN DE COLUMNAS Y TRANSFORMACIÓN** --> ***Punto 4 del Reto***
- Selección de las siguientes columnas con `select()`:
  - State
  - Account_Length
  - Area_Code
  - Phone
  - Intl_Plan
  - Vmail_Plan
  - Vmail_Message
- Creación de una nueva columna `Account_Length_MAS_1` resultado de sumar 1 a Account_Length con `col()`
- Todo se almacena en un nuevo dataframe que llamamos `datos_seleccionados`
- Se muestran los 10 primeros registros

In [11]:
from pyspark.sql.functions import col

datos_seleccionados_df = churn_df.select(
    "State",
    "Account_Length",
    "Area_Code",
    "Phone",
    "Intl_Plan",
    "Vmail_Plan",
    "Vmail_Message",
    (col("Account_Length") + 1).alias("Account_Length_MAS_1")
)

datos_seleccionados_df.show(10, truncate=False)

+-----+--------------+---------+--------+---------+----------+-------------+--------------------+
|State|Account_Length|Area_Code|Phone   |Intl_Plan|Vmail_Plan|Vmail_Message|Account_Length_MAS_1|
+-----+--------------+---------+--------+---------+----------+-------------+--------------------+
|PA   |163           |806      |403-2562|no       |yes       |300          |164                 |
|SC   |15            |836      |158-8416|yes      |no        |0            |16                  |
|MO   |131           |777      |896-6253|no       |yes       |300          |132                 |
|WY   |75            |878      |817-5729|yes      |yes       |700          |76                  |
|WY   |146           |878      |450-4942|yes      |no        |0            |147                 |
|VA   |83            |866      |454-9110|no       |no        |0            |84                  |
|IN   |140           |737      |331-5751|yes      |no        |0            |141                 |
|LA   |54           

**8. FILTRADO DE DATOS** --> ***`Punto 5 del Reto`***
- Usando `filter()` se seleccionan los registros que corresponden con State = New York (NY)
- Se guardan en un dataframe `datos_seleccionados_ny_df`
- Se muestran los 10 primeros registros seleccionados

In [12]:
datos_seleccionados_ny_df = churn_df.filter(col("State") == "NY")

print("Número de registros asociados a Nueva York:", datos_seleccionados_ny_df.count())
datos_seleccionados_ny_df.show(10, truncate=False)

Número de registros asociados a Nueva York: 98
+-----+--------------+---------+--------+---------+----------+-------------+------------------+---------+------------------+------------------+---------+------------------+------------------+-----------+------------------+------------------+----------+------------------+--------------+------+
|State|Account_Length|Area_Code|Phone   |Intl_Plan|Vmail_Plan|Vmail_Message|Day_Mins          |Day_Calls|Day_Charge        |Eve_Mins          |Eve_Calls|Eve_Charge        |Night_Mins        |Night_Calls|Night_Charge      |Intl_Mins         |Intl_Calls|Intl_Charge       |CustServ_Calls|Churn |
+-----+--------------+---------+--------+---------+----------+-------------+------------------+---------+------------------+------------------+---------+------------------+------------------+-----------+------------------+------------------+----------+------------------+--------------+------+
|NY   |142           |788      |248-9207|yes      |no        |0        

**9. PRIMERA AGREGACIÓN** --> ***`Punto 6 del Reto`***
> Obtención del número de registros para cada `estado`
- Primero agrupamos por estados usando `groupBy()`
- Contamos en cada agrupación con `count()`
- Renombramos la columna con `withColumnRenamed()`
- Ordenamos de mayor a menor `(desc)`
- Se guardan los datos en un dataframe que llamamos `datos_por_estado_df`
- Mostramos los 10 primeros registros del dataframe resultante

In [13]:
from pyspark.sql.functions import desc

datos_por_estado_df = (
    churn_df.groupBy("State")
    .count()
    .withColumnRenamed("count", "Cuenta")
    .orderBy(desc("Cuenta"))
)

datos_por_estado_df.show(10, truncate=False)

+-----+------+
|State|Cuenta|
+-----+------+
|RI   |120   |
|SC   |113   |
|MD   |113   |
|DC   |112   |
|ID   |111   |
|OH   |111   |
|MN   |110   |
|AR   |110   |
|NE   |109   |
|MO   |106   |
+-----+------+
only showing top 10 rows


**10. SEGUNDA AGREGACIÓN** --> ***`Punto 7 del Reto`***
> Obtención del número de registros por `Código de Área`
- Primero agrupamos por áreas usando `groupBy()`
- Contamos en cada agrupación con `count()`
- Renombramos la columna con `withColumnRenamed()`
- Ordenamos de mayor a menor (desc)
- Se guardan los datos en un dataframe que llamamos `datos_por_area`
- Mostramos los 10 primeros registros del dataframe resultante

In [14]:
from pyspark.sql.functions import desc

datos_por_area_df = (
    churn_df.groupBy("Area_Code")
    .count()
    .withColumnRenamed("count", "Cuenta")
    .orderBy(desc("Cuenta"))
)

datos_por_area_df.show(10, truncate=False)

+---------+------+
|Area_Code|Cuenta|
+---------+------+
|777      |317   |
|776      |291   |
|786      |284   |
|787      |278   |
|836      |215   |
|736      |214   |
|878      |207   |
|797      |204   |
|686      |203   |
|788      |197   |
+---------+------+
only showing top 10 rows


**11. TERCERA AGREGACIÓN** --> ***`Punto 8 del Reto`***
> Obtención de los minutos totales de llamadas diurnas en Indiana
- Filtramos los registros para el estado de Indiana que llamamos y sobre el filtrado sumamos los registros de la columna `Day_Mins` redondeando a 2 decimales el resultado con `sum()`
- La suma se almacena como `TOTAL_MIN_LLAMADAS_DIA_INDIANA` en un dataframe `sumas_llamadas_in_df`
- Mostramos la suma con `show()`, siendo el resultado mostrado la suma de todos los minutos de llamada durante el día en el estado de Indiana (IN)

In [17]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

suma_llamadas_in_df = churn_df.filter(col("State") == "IN").agg(
    spark_round(spark_sum("Day_Mins"), 2).alias("TOTAL_MIN_LLAMADAS_DIA_INDIANA")
)

suma_llamadas_in_df.show()

+------------------------------+
|TOTAL_MIN_LLAMADAS_DIA_INDIANA|
+------------------------------+
|                        518.33|
+------------------------------+



**12. CUARTA AGREGACIÓN** --> ***Punto 9 del Reto***
> Suma de todo el gasto de llamadas nocturnas `(Night_Charge)` en cada estado

- Agrupamos los registros por estados con  `groupBy()`
- Sumamos los registros de la columna `Night_Charge` redondeando a 2 decimales el resultado con `sum()`
- Le damos un alias para mostrar como `TOTAL_CARGO_LLAMADAS_NOCHE`
- Ordenamos de mmayor a menor
- Todo se guarda en un dataframe que llamamos `gasto_llamadas_noche_por_estado_df`
- Mostramos el resultado con `show()`, siendo el resultado mostrado la suma de el gasto de las llamadas nocturnas por estado

In [18]:
gasto_llamadas_noche_por_estado_df = (
    churn_df.groupBy("State")
    .agg(
        spark_round(
            spark_sum("Night_Charge"), 2
        ).alias("TOTAL_CARGO_LLAMADAS_NOCHE")
    )
    .orderBy(desc("TOTAL_CARGO_LLAMADAS_NOCHE"))
)

gasto_llamadas_noche_por_estado_df.show(truncate=False)

+-----+--------------------------+
|State|TOTAL_CARGO_LLAMADAS_NOCHE|
+-----+--------------------------+
|RI   |598.36                    |
|SC   |596.17                    |
|OH   |582.48                    |
|MD   |580.6                     |
|MN   |553.9                     |
|ID   |549.42                    |
|NE   |546.48                    |
|DC   |544.94                    |
|AR   |544.63                    |
|WV   |543.01                    |
|MS   |539.23                    |
|MO   |533.33                    |
|MA   |533.25                    |
|WY   |526.29                    |
|IA   |522.82                    |
|VT   |519.02                    |
|PA   |519.0                     |
|OR   |514.06                    |
|CA   |508.33                    |
|AL   |507.24                    |
+-----+--------------------------+
only showing top 20 rows
